In [ ]:
import sys
sys.path.append('..')

In [ ]:
import torch
import matplotlib.pyplot as plt

class CartesianSTFExtractor(torch.nn.Module):
    """
    Extracts STF features up to l_max from unit vectors rhats.

    Input:
        rhats: [..., 3]

    Output if padded=False:
        tuple of length l_max + 1
        outs[l].shape == [..., 2*l + 1]

    Output if padded=True:
        feats: [..., l_max + 1, 2*l_max + 1]
        mask:  [l_max + 1, 2*l_max + 1]
    """

    def __init__(self, l_max, pmaps, padded=False):
        super().__init__()
        self.l_max = l_max
        self.padded = padded

        n_monomials = 3**l_max
        n_channels = 2*l_max + 1

        # Every length-l_max Cartesian multi-index.  Degree-l monomials use
        # the first l coordinates, so all suffixes represent the same prefix.
        powers = torch.arange(l_max - 1, -1, -1)
        multi_index = (
            torch.arange(n_monomials)[:, None] // (3**powers)
        ) % 3

        pmap_dtype = torch.as_tensor(pmaps[0]).dtype
        if not torch.empty((), dtype=pmap_dtype).is_floating_point():
            pmap_dtype = torch.get_default_dtype()
        pmap = torch.zeros(l_max + 1, n_monomials, n_channels, dtype=pmap_dtype)
        mask = torch.zeros(l_max + 1, n_channels, dtype=torch.bool)

        # pmaps[l] is expected to have shape:
        #   l = 0: [1] or [1, 1]
        #   l = 1: [3, 3]
        #   l = 2: [3, 3, 5]
        #   l = 3: [3, 3, 3, 7]
        #   ...
        for l in range(l_max + 1):
            P = torch.as_tensor(pmaps[l]).reshape(3**l, 2*l + 1)
            prefix = torch.zeros(n_monomials, dtype=torch.long)
            if l > 0:
                prefix_powers = torch.arange(l - 1, -1, -1)
                prefix = (multi_index[:, :l] * (3**prefix_powers)).sum(dim=-1)

            pmap[l, :, :2*l + 1] = P[prefix] / (3 ** (l_max - l))
            mask[l, :2*l + 1] = True

        self.register_buffer("multi_index", multi_index)
        self.register_buffer("pmap", pmap)
        self.register_buffer("mask", mask)

    def forward(self, rhats):
        leading_shape = rhats.shape[:-1]
        x = rhats.reshape(-1, 3)

        B = x.shape[0]

        factors = x[:, self.multi_index]
        monomials = torch.cat(
            [
                x.new_ones(B, self.multi_index.shape[0], 1),
                factors.cumprod(dim=-1),
            ],
            dim=-1,
        )

        feat = torch.einsum(
            "bnl,lnc->blc",
            monomials,
            self.pmap.to(dtype=x.dtype),
        ).reshape(*leading_shape, self.l_max + 1, 2*self.l_max + 1)

        if not self.padded:
            return tuple(feat[..., l, :2*l + 1] for l in range(self.l_max + 1))

        return feat, self.mask

In [ ]:
l_max = 3
n_monomials = 3**l_max
n_channels = 2*l_max + 1

# Every length-l_max Cartesian multi-index.  Degree-l monomials use
# the first l coordinates, so all suffixes represent the same prefix.
powers = torch.arange(l_max - 1, -1, -1)
n_channels

In [ ]:

# set default dtype to float64 for better numerical stability
torch.set_default_dtype(torch.float64)
# use opt_einsum when it is available
try:
    import opt_einsum

    shared_intermediates = opt_einsum.shared_intermediates
except ImportError:
    from contextlib import nullcontext

    shared_intermediates = nullcontext
    
ndim = 3
import itertools
import collections
 

def pinv_project(projector):
    rank = len(projector.shape) - 1
    flattened = projector.reshape(3**rank, 2 * rank + 1)
    pinv = torch.linalg.pinv(flattened).T
    pinv = pinv.reshape(
        *(3,) * rank,
        2 * rank + 1,
    )
    return pinv

def tpl(int_tensor):
    return tuple(x.item() for x in int_tensor.unbind())

def cartesian_irreducible_mapping(ind_order):
    # First, construct the map from the full cartesian space to a symmetric basis,
    # mapping each element from the cartesian space to the sorted version of its indices.
    input_ind = []
    output_ind = []
    for ind in itertools.product(range(ndim), repeat=ind_order):
        input_ind.append(ind)
        output_ind.append(tuple(sorted(ind)))
    out_ind = torch.as_tensor(output_ind)
    in_ind = torch.as_tensor(input_ind)

    # List of symmetric indices only.
    needed_ind = torch.unique(out_ind, dim=0)

    # Figure out numbering for nonsymmetric indices within the symmetric ones.
    # # This step is of somewhat large size, could maybe be reduced with better algorithm.
    equal = (needed_ind == out_ind.unsqueeze(1)).all(dim=2)
    out_order = torch.where(equal)[1]

    # build sparse matrix -> all values are 1
    ind_comb = torch.stack(
        (torch.arange(len(out_order)), out_order),
    )
    xform = torch.sparse_coo_tensor(ind_comb, torch.ones(len(out_order)), dtype=torch.int64).to_dense()
    # This maps symmetric space to d^order space
    # shape (d^o, t(o)) where t(o) is the triangular number.

    # Now we need to build the map from traceless symmetric tensors to symmetric ones.

    # Which indices are constrained by tracelessness
    # Constrained elements are ones that end with 2,2 in the full cartesian space.
    # # Again, may use more memory than needed here.
    constrained_elements = (needed_ind[:, -2:] == torch.as_tensor([2, 2])).all(dim=-1)
    constrained_pos = torch.where(constrained_elements)[0]
    # Set of indices which are constrained by trace.
    constrained_ind = needed_ind[constrained_elements]

    # Which indices are not constrained
    free_elements = ~constrained_elements
    free_pos = torch.where(free_elements)[0]

    if ind_order == 1:
        # Exception to above which is wrong when only one dimension
        constrained_ind = []
        free_elements = [True, True, True]

    # Now we use regular python b/c it is easier to construct these loops.

    # Map from the index space to the symmetric basis number.
    bare_sym_map = {tpl(x): i for i, x in enumerate(needed_ind.unbind(0))}
    # Map from the index space to traceless symmetric baseless number, but only for values
    # that are in both (i.e. not affected by tracelessness)
    bare_traceless_map = {tpl(x): i for i, x in enumerate(needed_ind[free_elements].unbind(0))}

    # Map from symmetric indices to traceless ones, but only for values that are in both.
    sym_traceless_map = {bare_sym_map[x]: j for x, j in bare_traceless_map.items()}

    # Initialize full map in "csr" form based on unconstrained elements.
    sym_traceless_csr = collections.defaultdict(list)
    for k, v in sym_traceless_map.items():
        sym_traceless_csr[k].append((v, 1))

    # Extend map for trace-constrained elements
    for x in map(tpl, constrained_ind):
        x_sym = bare_sym_map[x]
        new_sym_vals = []
        # Get the two other components x...00 and x_...11 (constraint is on x...22)
        for k in (0, 1):
            a = tuple(sorted(x[:-2] + (k, k)))
            aa = bare_sym_map[a]
            new_sym_vals.append(aa)

        # Write this is in terms of
        new_traceless_vals = collections.Counter()
        for aa in new_sym_vals:
            for aaa, v in sym_traceless_csr[aa]:
                new_traceless_vals[aaa] += -v
        new_traceless_vals = [(k, v) for k, v in new_traceless_vals.items()]
        sym_traceless_csr[x_sym].extend(new_traceless_vals)

    # Rewrite in coo form
    sym_traceless_coo = collections.Counter()
    for i, row in sym_traceless_csr.items():
        for j, val in row:
            sym_traceless_coo[i, j] += val

    # Convert into pytorch tensor
    ind, vals = zip(*list(sym_traceless_coo.items()))
    ind = torch.as_tensor(ind).T
    vals = torch.as_tensor(vals)
    xform2 = torch.sparse_coo_tensor(ind, vals, size=(xform.shape[1], 2 * ind_order + 1))

    # The full map is simply the map (cartesian index, symmetric) @ (symmetric, sym. traceless)
    full_xform = xform @ xform2.to_dense()

    # unflatten cartesian set.
    full_xform = full_xform.reshape(*((ndim,) * ind_order), -1)

    return full_xform

c = {}
p = {}
c[0] = torch.ones((1, 1))
p[0] = torch.ones((1, 1))
for i in range(1, 5):
    c[i] = cartesian_irreducible_mapping(i).to(torch.get_default_dtype())
    p[i] = pinv_project(c[i])
cmaps = c
pmaps = p

class TensorExtractor(torch.nn.Module):
    def __init__(self, l_max, pmaps=pmaps):
        super().__init__()
        self.l_max = l_max
        all_pmaps = list(pmaps.values())
        self.pmaps = torch.nn.ParameterList(all_pmaps[: l_max + 1])
        for p in self.pmaps:
            p.requires_grad_(False)
    def forward(self, rhats):
        s = v = q = t = f = None
        s = torch.ones(rhats.shape[0], device=rhats.device, dtype=rhats.dtype).unsqueeze(1)
        with shared_intermediates():  # opt_einsum or null, see import logic
            if self.l_max > 0:
                v = rhats
            if self.l_max > 1:
                q = torch.einsum("ijk,bi,bj->bk", self.pmaps[2], rhats, rhats)
            if self.l_max > 2:
                t = torch.einsum("ijkl,bi,bj,bk->bl", self.pmaps[3], rhats, rhats, rhats)
            if self.l_max > 3:
                f = torch.einsum("ijklm,bi,bj,bk,bl->bm", self.pmaps[4], rhats, rhats, rhats, rhats)
        return s, v, q, t, f


In [ ]:
def sample_unit_circle_points(n, device=None, ntype=torch.float64):
    theta = torch.arange(n, device=device, dtype=ntype) * (2 * torch.pi / n)
    points = torch.zeros(n, 3, device=device, dtype=ntype)
    points[:, 0] = torch.cos(theta)
    points[:, 1] = torch.sin(theta)
    return points


def plot_points(points, colors=None, limits=(-1.5, 1.5), plot_middle=True):
    import matplotlib.pyplot as plt

    points = points.detach().cpu()
    views = [(30, -60), (90, -90), (0, 0)]
    # Turn of ticks for orthogonal axis
    fig = plt.figure(figsize=(12, 4))

    for i, (elev, azim) in enumerate(views, start=1):
        ax = fig.add_subplot(1, 3, i, projection="3d")
        if colors is not None:
            ax.scatter(points[:, 0], points[:, 1], points[:, 2], s=100, c=colors)
        else:       
            ax.scatter(points[:, 0], points[:, 1], points[:, 2], s=100)
        if plot_middle:
            ax.scatter(0, 0, 0, s=50, c="red", marker="x")
        if i == 2: 
            ax.set_zticks([])
        else:
            ax.set_xticks([])
        ax.view_init(elev=elev, azim=azim)
        ax.set_xlim(limits)
        ax.set_ylim(limits)
        ax.set_zlim(limits)

    plt.show()

def rotation_matrix_3d(axis, angle, device=None, dtype=torch.get_default_dtype()):
    axis = torch.as_tensor(axis, device=device, dtype=dtype)
    axis = axis / axis.norm()

    x, y, z = axis
    c = torch.cos(torch.as_tensor(angle, device=device, dtype=dtype))
    s = torch.sin(torch.as_tensor(angle, device=device, dtype=dtype))
    C = 1 - c

    return torch.tensor(
        [
            [c + x*x*C,     x*y*C - z*s, x*z*C + y*s],
            [y*x*C + z*s,   c + y*y*C,   y*z*C - x*s],
            [z*x*C - y*s,   z*y*C + x*s, c + z*z*C],
        ],
        device=device,
        dtype=dtype,
    )


n_points = 2
def produce_pattern(rotation_angle=torch.pi / 8, fold1=2, fold2=3):
    points_2_fold = sample_unit_circle_points(fold1) @ rotation_matrix_3d([0, 0, 1], rotation_angle)
    colors_2_fold = ['blue'] * fold1
    points_3_fold = sample_unit_circle_points(fold2) 
    colors_3_fold = ['orange'] * fold2
    points = torch.cat([points_2_fold, points_3_fold], dim=0)
    colors_joined = [*colors_2_fold, *colors_3_fold]
    return points, colors_joined

fold1 = 2
fold2 = 3
points, colors_joined = produce_pattern(rotation_angle=10, fold1=fold1, fold2=fold2)

angle_samples = torch.linspace(15, torch.pi, steps=120, dtype=torch.get_default_dtype())
# TODO: Sample rotation and calculate distance between the two sets of points. Then, use the extractor to compute features and compare them.
points_set = []
for angle in angle_samples:
    points_rotated, _ = produce_pattern(rotation_angle=angle, fold1=fold1, fold2=fold2)
    points_set.append(points_rotated)
points_set = torch.stack(points_set, dim=0)  # shape: [360, fold1+fold2, 3]
plot_points(points_set[0], colors=colors_joined)
plot_points(points_set[-1], colors=colors_joined)


In [ ]:
three_folds = points_set[:, fold1:, :]  # shape: [360, fold2, 3]
two_folds = points_set[:, :fold1, :]  # shape: [360, fold1, 3]

# Calculate the distance between the two sets of points for each rotation
distances = torch.cdist(two_folds, three_folds, p=2)
min_distance, _ = torch.min(distances.view(-1, fold1 * fold2), dim=1)
print(min_distance.shape)  # Should be [360]
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(angle_samples, min_distance)
ax.set_title(f"Minimum distance between {fold1}-fold and {fold2}-fold points as a function of rotation angle")
ax.set_xlabel("Rotation Angle")
# Ticks for angles
ax.set_ylabel("Minimum Distance")

In [ ]:
hippynnn = TensorExtractor(l_max=5)
s, v, Q, T, F = hippynnn(points)
# Sum over deltas -> They can be multiplied with different weights
s0, s1, s2, s3, s4 = s.sum(dim=0), v.sum(dim=0), Q.sum(dim=0), T.sum(dim=0), F.sum(dim=0)
f0, f1, f2, f3, f4 = hippynnn.pmaps[0] @ s0.T, hippynnn.pmaps[1] @ s1.T, hippynnn.pmaps[2] @ s2, hippynnn.pmaps[3] @ s3, hippynnn.pmaps[4] @ s4 

torch.all( torch.isclose(f0, torch.zeros_like(f0), atol=1e-6) ), torch.all( torch.isclose(f1, torch.zeros_like(f1), atol=1e-6) ), torch.all( torch.isclose(f2, torch.zeros_like(f2), atol=1e-6) ), torch.all( torch.isclose(f3, torch.zeros_like(f3), atol=1e-6) )

In [ ]:
import matplotlib.pyplot as plt
from notebooks.contraction_formats import graph_from_formula, einsum_to_formula
from notebooks.contraction_visualization import draw_graph_example
HIPHOP_l3_n4 = [
    "i, i -> ", 
    "ij, ij -> ", 
    "ij, ik, jk -> ",
    "ijk, ijk -> ",
    "ijk, ijl, kmn, lmn -> ", 
    "i, ij, j -> ", 
    "i, ij, jk,k -> ", 
    "i, j, k, ijk -> ",
    "i, ijk, jkl, l ->", 
    "ij, ikl, klj ->", 
    "ij,ik, kml, lmj ->",
    "ij,kl,ijm,klm ->", 

    # Crazy guy 
    "ijk,lmn,il,jm,kn->"
    ]

HIPHOP_l4_n4 = [
    *HIPHOP_l3_n4,
    "ijkl, ijkl -> ",
    "imjp, iklm, kljp ->", 
    "ik, ikjl, jl -> ", 
    "ij, iklm, kljm ->", 
    "ijo,iko, kmlp, lmjp ->",
    "ijp,iko, kmlo, lmjp ->",
    "ljp,iko, kmio, lmjp ->",
    "abc,def,ghi,jkl,ihck,gfda,lebj->",
]

BASIS = HIPHOP_l3_n4

fig, axs = plt.subplots(len(BASIS)//4 + 1, 4, figsize=(12,16))
flattened_axs = axs.flatten()


for i, einsum in enumerate(BASIS):
    formula = einsum_to_formula(einsum)
    graph = graph_from_formula(formula)
    draw_graph_example(ax=flattened_axs[i], G=graph, title=einsum)
fig.tight_layout()
plt.savefig("hiphop_l3_n4_graphs.png", dpi=300)

In [ ]:
# Calculate contraction 
def contract_tensors(tensor_dict, einsum_str):
    """
    Contract a list of tensors according to the provided einsum string.

    Args:
        tensor_dict (dict): A dictionary mapping tensor names to torch.Tensor objects.
        einsum_str (str): The einsum string specifying the contraction.

    Returns:
        torch.Tensor: The result of the contraction.
    """
    # Get tensors from einsum string
    tensor_idx = [len(s.strip()) for s in einsum_str.split('->')[0].split(',')]
    tensors = [tensor_dict[i] for i in tensor_idx]

    return torch.einsum(einsum_str, *tensors)

def calc_basis(tensor_dict, einsum_basis):
    """
    Calculate the basis for a given contraction of tensors.

    Args:
        tensor_dict (dict): A dictionary mapping tensor names to torch.Tensor objects.
        einsum_basis (str): The einsum string specifying the contraction.
    Returns:
        torch.Tensor: The result of the contraction.
    """
    invariants = torch.zeros(len(einsum_basis), dtype=torch.float64)
    for i, einsum in enumerate(einsum_basis):
        result = contract_tensors(tensor_dict, einsum)
        invariants[i] = result
    return invariants

def get_nonzeros(tensors, basis):
    invariants = calc_basis(tensors, basis)
    nonzero_idx = torch.where(torch.abs(invariants).reshape(-1) > 1e-6)[0]
    print(f"{nonzero_idx}")
    return invariants[nonzero_idx], [basis[i] for i in nonzero_idx]

mappings = [cartesian_irreducible_mapping(i).to(torch.get_default_dtype()) for i in range(1, 5)]
def calculate_tensors(points): 
    tensors = hippynnn(points)
    if len(tensors) == 4:
        s, v, Q, T = tensors
        s0, s1, s2, s3 = s.sum(dim=0), v.sum(dim=0), Q.sum(dim=0), T.sum(dim=0)
        f0, f1, f2, f3 = s0, mappings[0] @ s1, mappings[1] @ s2, mappings[2] @ s3
        return f0, f1, f2, f3
    elif len(tensors) == 5: 
        s, v, Q, T, F = tensors
        s0, s1, s2, s3, s4 = s.sum(dim=0), v.sum(dim=0), Q.sum(dim=0), T.sum(dim=0), F.sum(dim=0)
        f0, f1, f2, f3, f4 = s0, mappings[0] @ s1, mappings[1] @ s2, mappings[2] @ s3, mappings[3] @ s4
        return f0, f1, f2, f3, f4
    else:
        raise ValueError(f"Unexpected number of tensors returned: {len(tensors)}")

In [ ]:

    
invariants_versus_angle = torch.zeros(points_set.shape[0], len(BASIS))
prev_f3 = None
prev_f2 = None
for points_idx, points in enumerate(points_set):
    fs = calculate_tensors(points)
    tensors = {idx: f for idx, f in enumerate(fs)}
    #if prev_f4 is not None:
    #   assert torch.allclose(fs[4], prev_f4, atol=1e-10), f"f4 changed at index {points_idx}"
    #prev_f4 = fs[4]
    if prev_f3 is not None:
       assert torch.allclose(fs[3], prev_f3, atol=1e-10), f"f3 changed at index {points_idx}"
    #print(f"f4: {fs[4]}")
    prev_f3 = fs[3]
    if prev_f2 is not None:
       assert not torch.allclose(fs[2], prev_f2, atol=1e-10), f"f2 does not change at index {points_idx}"
    prev_f2 = fs[2]
    invariants = calc_basis(tensors, BASIS)
    invariants_versus_angle[points_idx] = invariants

In [ ]:
non_zero_idx = torch.where(torch.isclose(invariants_versus_angle, torch.zeros_like(invariants_versus_angle), atol=1e-6).sum(dim=0) == 0)[0]
non_zero_basis = [BASIS[i] for i in non_zero_idx]
non_zero_invariants = invariants_versus_angle[:, non_zero_idx]
# normalization 
degrees = [len(b.split('->')[0].split(','))-1 for b in non_zero_basis]
degrees = torch.as_tensor(degrees, dtype=torch.float64)
R = torch.sqrt(invariants_versus_angle[:, 0:1] + invariants_versus_angle[:, 1:2] + invariants_versus_angle[:, 3:4])
normed_invariants = non_zero_invariants / torch.clamp(R ** degrees, min=1e-8)

fig, axs = plt.subplots(2, len(non_zero_basis), figsize=(4*len(non_zero_basis), 8))
for i, einsum in enumerate(non_zero_basis):
    formula = einsum_to_formula(einsum)
    graph = graph_from_formula(formula)
    draw_graph_example(ax=axs[0, i], G=graph, title=f"{einsum}")
    axs[1, i].plot(angle_samples, non_zero_invariants[:, i]) 
    #axs[1,i].plot(angle_samples, normed_invariants[:, i], label="Normalized")
    axs[1, i].set_xlabel("Rotation Angle")
    axs[1, i].set_ylabel("Invariant Value")
    axs[1, i].set_ylim([min(torch.min(non_zero_invariants[:, i]) - 0.1, torch.min(normed_invariants[:, i]) - 0.1), max(torch.max(non_zero_invariants[:, i]) + 0.1, torch.max(normed_invariants[:, i]) + 0.1)])

In [ ]:
import numpy as np
logan_points_label0 = np.load("/Users/karella/Projects/rotation-invariant-neural-networks/notebooks/2d_ring_graphs__seed0_10/")['positions'] 
logan_points_label0 = torch.as_tensor(logan_points_label0, dtype=torch.get_default_dtype())[1:, : ]

plot_points(logan_points_label0, colors=['blue', 'blue', 'orange', 'orange', 'orange'])
norm_points = logan_points_label0 / logan_points_label0.norm(dim=-1, keepdim=True)
plot_points(norm_points, colors=['blue', 'blue', 'orange', 'orange', 'orange'])

logan_points_label1 = np.load("/Users/karella/Projects/rotation-invariant-neural-networks/notebooks/2d_ring_graphs__seed0_10/2_inner_3_outer/graphs/graph_0009_label1_far.npz")['positions']
logan_points_label1 = torch.as_tensor(logan_points_label1, dtype=torch.get_default_dtype())[1:, : ]
plot_points(logan_points_label1, colors=['blue', 'blue', 'orange', 'orange', 'orange'])
norm_points1 = logan_points_label1 / logan_points_label1.norm(dim=-1, keepdim=True)
plot_points(norm_points1, colors=['blue', 'blue', 'orange', 'orange', 'orange'])

In [ ]:
from glob import glob
# calculate invariants for logan_points_label0
all_label_0 = sorted(glob("/Users/karella/Projects/rotation-invariant-neural-networks/notebooks/2d_ring_graphs__seed0_10/2_inner_3_outer/graphs/*label0*.npz"))
radius_weights = torch.tensor([1.0, 1.0, 1.0, 1.0, 1.0]).view((5, 1))

invariants_label0 = torch.zeros(len(all_label_0), len(HIPHOP_l3_n4), dtype=torch.get_default_dtype())

for i, file in enumerate(all_label_0):
    logan_points_label0 = np.load(file)['positions']
    norm_points = torch.as_tensor(logan_points_label0, dtype=torch.get_default_dtype())[1:, : ]
    norm_points = norm_points / norm_points.norm(dim=-1, keepdim=True)
    f0, f1, f2, f3 = calculate_tensors(norm_points * radius_weights)
    label0 = calc_basis({0: f0, 1: f1, 2: f2, 3: f3}, HIPHOP_l3_n4)
    invariants_label0[i] = label0

all_label_1 = sorted(glob("/Users/karella/Projects/rotation-invariant-neural-networks/notebooks/2d_ring_graphs__seed0_10/2_inner_3_outer/graphs/*label1*.npz"))

invariants_label1 = torch.zeros(len(all_label_1), len(HIPHOP_l3_n4), dtype=torch.get_default_dtype())

for i, file in enumerate(all_label_1):
    logan_points_label1 = np.load(file)['positions']
    norm_points1 = torch.as_tensor(logan_points_label1, dtype=torch.get_default_dtype())[1:, : ] 
    norm_points1 = norm_points1 / norm_points1.norm(dim=-1, keepdim=True)
    f0, f1, f2, f3 = calculate_tensors(norm_points1 * radius_weights)
    label1 = calc_basis({0: f0, 1: f1, 2: f2, 3: f3}, HIPHOP_l3_n4)
    invariants_label1[i] = label1



In [ ]:
# Test the invariance
pattern =  points_set[0:1]
angles = torch.linspace(0, 2 * torch.pi, steps=360, dtype=torch.get_default_dtype())

pattern

In [ ]:
# Plot the invariants for label 0 and label 1
fig, axs = plt.subplots(2, len(HIPHOP_l3_n4), figsize=(4*len(HIPHOP_l3_n4), 8))
for i, einsum in enumerate(HIPHOP_l3_n4):
    formula = einsum_to_formula(einsum)
    graph = graph_from_formula(formula)
    draw_graph_example(ax=axs[0, i], G=graph, title=f"{einsum}")
    bp = axs[1, i].boxplot(
    [invariants_label0[:, i], invariants_label1[:, i]],
    tick_labels=["Label 0", "Label 1"],
    patch_artist=True,
    showmeans=True,
)

    for box, color in zip(bp["boxes"], ["tab:blue", "tab:orange"]):
        box.set_facecolor(color)
        box.set_alpha(0.75)

    for median in bp["medians"]:
        median.set_color("black")

    axs[1, i].set_ylabel("Invariant Value")
    axs[1, i].set_title(f"Invariant {i}")

    axs[1, i].set_xlabel("Sample Index")
    axs[1, i].set_ylabel("Invariant Value")
    axs[1, i].legend()

In [ ]:
import numpy as np
logan_points_label0 = np.load("/Users/karella/Projects/rotation-invariant-neural-networks/notebooks/2d_ring_graphs__seed0_10/2_inner_3_outer/graphs/graph_0000_label0_close.npz")['positions'][1:, : ]
logan_points_label1 = np.load("/Users/karella/Projects/rotation-invariant-neural-networks/notebooks/2d_ring_graphs__seed0_10/2_inner_3_outer/graphs/graph_0009_label1_far.npz")['positions'][1:, : ]

# Plot those
plot_points(torch.as_tensor(logan_points_label0, dtype=torch.get_default_dtype()), colors=['blue', 'blue', 'orange', 'orange', 'orange'])
plot_points(torch.as_tensor(logan_points_label1, dtype=torch.get_default_dtype()), colors=['blue', 'blue', 'orange', 'orange', 'orange'])

In [ ]:
import matplotlib.pyplot as plt

from ase import Atoms
from ase.visualize.plot import plot_atoms
from ase.io import read

atomsA = read("bond_like_orient.xyz")
atomsB = read("non_bond_like_orient.xyz")
# randomly rotate the atoms
fig, ax = plt.subplots()
plot_atoms(atomsA, ax)
ax.set_axis_off()
plt.show()

In [ ]:
plot_points(points=torch.from_numpy(atomsA.arrays['positions']),
            colors=['red' if num == 8 else 'blue' for num in torch.from_numpy(atomsA.arrays['numbers'])],
            limits=(-5, 5),
            plot_middle=False)

# rotate by 90 degrees around z-axis
rotation_matrix = rotation_matrix_3d([0, 1 , 0], torch.pi / 2)
atomsA_copy = atomsA.copy()
# rotate by 90 degrees around z-axis
atomsA_copy.arrays['positions'] = atomsA_copy.arrays['positions'] @ rotation_matrix.T.numpy()

rotated_positions = torch.from_numpy(atomsA_copy.arrays['positions'])
plot_points(points=rotated_positions,
            colors=['red' if num == 8 else 'blue' for num in torch.from_numpy(atomsA_copy.arrays['numbers'])],
                    limits=(-5, 5),
            plot_middle=False)



In [ ]:
# Fragments -> calculate tensors for each of the fragments and then combine them. 
# Random rotation 

#atomsA.rotate(45, 'z', rotate_cell=True)

frag1A, clrs1A = atomsA.positions[:3, :], atomsA.arrays['numbers'][:3]
frag2A, clrs2A = atomsA.positions[3:, :], atomsA.arrays['numbers'][3:]
distA = frag1A[0, :] - frag2A[0, :]

frag1B, clrs1B = atomsB.positions[:3, :], atomsB.arrays['numbers'][:3]
frag2B, clrs2B = atomsB.positions[3:, :], atomsB.arrays['numbers'][3:]
distB = frag1B[0, :] - frag2B[0, :]

# Center the fragment 
frag2A = frag2A - frag2A[:1, :]
frag2B = frag2B - frag2B[:1, :]


def _preprocess(positions, numbers): 
    frag1, clrs1 = positions[:3, :], numbers[:3]
    frag2,  clrs2 = positions[3:, :], numbers[3:]
    dist = frag1[0, :] - frag2[0, :]
    # Move to zero 
    frag2 = frag2 - frag2[:1, :]

    assert clrs1[0] == 8 and clrs2[0] == 8, "First atom in each fragment should be oxygen (atomic number 8)."
    assert torch.allclose(frag1[0], torch.zeros(3)) and torch.allclose(frag2[0], torch.zeros(3)), "First atom in each fragment should be at the origin."
    
    # normalize the points to unit sphere
    frag1 = frag1[1:] / torch.norm(frag1[1:], dim=-1, keepdim=True)
    frag2 = frag2[1:] / torch.norm(frag2[1:], dim=-1, keepdim=True)

    return frag1, frag2


a, b = _preprocess(torch.from_numpy(atomsA.positions), torch.tensor(atomsA.arrays['numbers']))
A, B = _preprocess(torch.from_numpy(atomsA_copy.positions), torch.tensor(atomsA_copy.arrays['numbers']))


def calculate_tensors(rhats):
    # Check all the norms
    assert torch.allclose(rhats.norm(dim=-1), torch.ones(rhats.shape[0])), "Input vectors must be unit vectors."
    s, v, Q, T = hippynnn(rhats)
    s0, s1, s2, s3 = s.sum(dim=0), v.sum(dim=0), Q.sum(dim=0), T.sum(dim=0)
    f0, f1, f2, f3 = hippynnn.pmaps[0] @ s0, hippynnn.pmaps[1] @ s1, hippynnn.pmaps[2] @ s2, hippynnn.pmaps[3] @ s3 
    return f0, f1, f2, f3

def calculate_invariants(f0, f1, f2, f3):
    tensors = {0: f0, 1: f1, 2: f2, 3: f3}
    return calc_basis(tensors, HIPHOP_l3_n4)

a_f0, a_f1, a_f2, a_f3 = calculate_tensors(a)
b_f0, b_f1, b_f2, b_f3 = calculate_tensors(b)
A_f0, A_f1, A_f2, A_f3 = calculate_tensors(A)
B_f0, B_f1, B_f2, B_f3 = calculate_tensors(B) 

# Check what tensors are equal for all the fragments.
print("f0A vs f0B vs f0A: ", torch.allclose(a_f0, b_f0) and torch.allclose(a_f0, A_f0, atol=1e-6), "... vs f0B: ", torch.allclose(a_f0, B_f0, atol=1e-6))
print("f1A vs f1B vs f1A: ", torch.allclose(a_f1, b_f1) and torch.allclose(a_f1, A_f1, atol=1e-6), "... vs f1B: ", torch.allclose(a_f1, B_f1, atol=1e-6))
print("f2A vs f2B vs f2A: ", torch.allclose(a_f2, b_f2) and torch.allclose(a_f2, A_f2, atol=1e-6), "... vs f2B: ", torch.allclose(a_f2, B_f2, atol=1e-6))
print("f3A vs f3B vs f3A: ", torch.allclose(a_f3, b_f3) and torch.allclose(a_f3, A_f3, atol=1e-6), "... vs f3B: ", torch.allclose(a_f3, B_f3, atol=1e-6))

# calculate invariants for the fragments and compare them.
a_inv = calculate_invariants(a_f0, a_f1, a_f2, a_f3)
b_inv = calculate_invariants(b_f0, b_f1, b_f2, b_f3)
A_inv = calculate_invariants(A_f0, A_f1, A_f2, A_f3)
B_inv = calculate_invariants(B_f0, B_f1, B_f2, B_f3)

# All invariants equal 
print("All invariants equal: ", torch.allclose(a_inv, b_inv) and torch.allclose(a_inv, A_inv, atol=1e-6) and torch.allclose(a_inv, B_inv, atol=1e-6))

In [ ]:
fold1 = 4
fold2 = 5
points, colors_joined = produce_pattern(rotation_angle=0, fold1=fold1, fold2=fold2)

In [ ]:
import numpy as np
from scipy.spatial.transform import Rotation as R

angles = 360
def random_rotation_matrices(seed=None):
    rng = np.random.default_rng(seed)
    while True:
        yield torch.from_numpy(R.random(rng=rng).as_matrix())

matrices = random_rotation_matrices(seed=42)


points, clrs = produce_pattern(rotation_angle=95, fold1=3, fold2=2)

randomly_rotated_points = [points]
rot_matrices = [torch.eye(3, dtype=torch.get_default_dtype())]

for i in range(angles):
    rot_matrix = next(matrices)
    rotated_points = points @ rot_matrix.T
    randomly_rotated_points.append(rotated_points)
    rot_matrices.append(rot_matrix)
randomly_rotated_points = torch.stack(randomly_rotated_points, dim=0)  # shape: [360, fold1+fold2, 3]

In [ ]:
# Plot the configuration of the first randomly rotated points
plot_points(randomly_rotated_points[0], colors=clrs, limits=(-1.5, 1.5), plot_middle=True)
# Last
plot_points(randomly_rotated_points[-1], colors=clrs, limits=(-1.5, 1.5), plot_middle=True)

In [ ]:
s1s = []
Qs = []
Qs_M = []
randomly_rotated_invariants = []
invs = []

# Calculate the tensors
for ps in randomly_rotated_points:
    f0, f1, f2, f3 = calculate_tensors(ps)

    tensors = {0: f0, 1: f1, 2: f2, 3: f3}
    invariants = calc_basis(tensors, HIPHOP_l3_n4)
    invs.append(invariants)
    randomly_rotated_invariants.append(invariants)
invariants = torch.stack(randomly_rotated_invariants, dim=0)  # shape: [360, num_invariants]
# All columns equal for all the rotations
print("All invariants equal for all rotations: ", torch.allclose(invariants, invariants[0:1, :], atol=1e-6))

In [ ]:
import torch


def invariant_basis_fast(v, Q, T):
    """
    v: (..., 3)
    Q: (..., 3, 3)      symmetric trace-free rank 2
    T: (..., 3, 3, 3)   fully symmetric trace-free rank 3

    Returns:
        (..., 13), in the same order as your listed basis.
    """

    # ---- shared Q intermediates ----
    Qv = torch.einsum("...ij,...j->...i", Q, v)
    Q2 = Q @ Q

    # ---- shared T:T matrix ----
    # B_ij = T_iab T_jab
    B = torch.einsum("...iab,...jab->...ij", T, T)
    Bv = torch.einsum("...ij,...j->...i", B, v)

    # ---- T(v,v)_i ----
    Tv = torch.einsum("...ijk,...k->...ij", T, v)
    Tvv = torch.einsum("...ij,...j->...i", Tv, v)

    # ---- w_m = Q_ij T_ijm = T:Q ----
    w = torch.einsum("...ij,...ijm->...m", Q, T)

    # ---- invariant: ij,ikm,kl,lmj -> ----
    # For symmetric T, each slice A_m = T[m,:,:] is equivalent to T[:,:,m].
    # This computes sum_m tr(A_m Q A_m Q).
    AQ = torch.matmul(T, Q.unsqueeze(-3))
    QTQT = torch.einsum("...mij,...mji->...", AQ, AQ)

    # ---- invariant: ijk,lmn,il,jm,kn -> ----
    # U_ijk = Q_il Q_jm Q_kn T_lmn
    U = torch.einsum("...lmn,...il->...imn", T, Q)
    U = torch.einsum("...imn,...jm->...ijn", U, Q)
    U = torch.einsum("...ijn,...kn->...ijk", U, Q)

    out = torch.stack(
        [
            # "i, i -> "
            (v * v).sum(dim=-1),

            # "ij, ij -> "
            (Q * Q).sum(dim=(-2, -1)),

            # "ij, ik, jk -> "
            # = tr(Q^3) for symmetric Q
            (Q2 * Q).sum(dim=(-2, -1)),

            # "ijk, ijk -> "
            (T * T).sum(dim=(-3, -2, -1)),

            # "ijk, ijl, kmn, lmn -> "
            # = tr(B^2) for fully symmetric T
            (B * B).sum(dim=(-2, -1)),

            # "i, ij, j -> "
            (v * Qv).sum(dim=-1),

            # "i, ij, jk, k -> "
            # = |Qv|^2 for symmetric Q
            (Qv * Qv).sum(dim=-1),

            # "i, j, k, ijk -> "
            (v * Tvv).sum(dim=-1),

            # "i, ijk, jkl, l -> "
            # = v^T B v for symmetric T
            (v * Bv).sum(dim=-1),

            # "ij, ikl, klj -> "
            # = Q:B for symmetric T
            (Q * B).sum(dim=(-2, -1)),

            # "ij,ikm, kl, lmj -> "
            QTQT,

            # "ij,kl,ijm,klm -> "
            # = |T:Q|^2
            (w * w).sum(dim=-1),

            # "ijk,lmn,il,jm,kn->"
            (T * U).sum(dim=(-3, -2, -1)),
        ],
        dim=-1,
    )

    return out